## 💡 Tip — 기능 발견

어떤 기능이 있는지 궁금할 때:

```python
app.help()        # 18개 accessor + context manager 목록
repr(app)         # 현재 상태 요약 — App(visible=True, version='13.0.0', docs=2, page=5/20)
```

전체 accessor 투어 → [10_accessors_overview.ipynb](10_accessors_overview.html)


한컴 오피스는 **액션(Action)** 이라는 개념으로 다양한 문서 편집을 가능하게 합니다. hwpapi 는 이 액션들을 파이썬에서 보다 쉽게 사용할 수 있도록 감싼 라이브러리입니다.

기본적인 컨셉은 `win32com` 으로 직접 호출하던 패턴을 간결한 메소드로 제공하는 것입니다.

이 문서에서 다루는 기본기:

1. App 객체 생성
2. 액션과 파라미터셋
3. 텍스트 입력
4. 커서 이동
5. 영역 선택
6. 서식 변경
7. 텍스트 가져오기

## 1. App 객체 — HWP 연결

`App` 은 hwpapi 의 진입점입니다. 인스턴스를 생성하는 순간 한글이 실행되거나(이미 실행 중이면) 기존 인스턴스에 연결됩니다.

```python
from hwpapi.core import App

app = App()                  # 한컴 오피스 연결/실행
app = App(is_visible=True)   # 화면에 창을 띄우기 (기본값 True)
app = App(is_visible=False)  # 백그라운드 실행 (대량 처리 시 권장)
```

::: {.callout-note}
대부분의 기능은 `app.actions.*` 와 `app.api.*` 로 접근합니다. 

- `app.actions` — 자주 쓰는 기능을 pythonic 메소드로 제공
- `app.api` — win32com 의 원시 `HwpObject` 그대로 — 필요하면 언제든 내려가서 사용
:::

## 2. 액션과 파라미터셋

한글의 편집 기능은 대부분 **액션** 으로 노출됩니다. 액션은 크게 두 가지로 나뉩니다.

| 종류 | 설명 | 예 |
|:---|:---|:---|
| **파라미터 없는 액션** | 별도 설정 없이 실행 | `Delete`, `BreakPara`, `SelectAll` |
| **파라미터 있는 액션** | ParameterSet 을 먼저 구성 | `InsertText`, `CharShape`, `TableCreate` |

```python
# 파라미터 없는 액션 — 바로 실행
app.actions.Delete.run()
app.actions.SelectAll.run()
app.actions.Undo.run()

# 파라미터 있는 액션 — pset 설정 후 실행
action = app.actions.InsertText
action.pset.Text = "입력할 텍스트"
action.run()
```

hwpapi 는 704 개 액션을 자동 등록합니다. 전체 목록:

```python
app.actions.list_actions()                    # 전체
app.actions.list_actions(with_pset_only=True) # pset 이 필요한 것만
```

## 3. 텍스트 입력

자주 쓰는 작업은 메소드로 감싸져 있습니다.

```python
app.insert_text("안녕하세요\n")
app.insert_text("한 번에 여러 줄도\n가능합니다.\n")

# 서식과 함께 입력
app.insert_text("큰 글자", height=2400)
```

내부 동작이 궁금하다면 ― 위 한 줄은 다음과 동일합니다.

```python
action = app.actions.InsertText
action.pset.Text = "안녕하세요\n"
action.run()
```

## 4. 커서 이동

hwpapi 는 `app.move` 접근자로 커서 이동 헬퍼를 제공합니다.

```python
app.move.top_of_file()       # 문서 처음으로
app.move.bottom_of_file()    # 문서 끝으로
app.move.current_list(para=5, pos=0)  # 5번째 문단 시작
```

그 외의 세밀한 이동은 액션으로:

```python
app.api.Run("MoveLineBegin")   # 현재 줄 시작
app.api.Run("MoveLineEnd")     # 현재 줄 끝
app.api.Run("MoveLineUp")      # 위 줄로
app.api.Run("MoveLineDown")    # 아래 줄로
app.api.Run("MoveLeft")        # 한 글자 왼쪽
app.api.Run("MoveRight")       # 한 글자 오른쪽
```

## 5. 영역 선택

서식을 바꾸거나 삭제하려면 영역을 먼저 선택해야 합니다.

```python
app.select_text()          # 현재 문단 선택
app.actions.SelectAll.run()  # 문서 전체 선택

# 특정 텍스트 찾아서 선택
app.find_text("검색어")     # 검색어 위치로 커서 이동 + 선택
```

선택 영역만 빼서 확인:

```python
selected = app.get_selected_text()
print(selected)
```

## 6. 서식 변경

선택한 영역(또는 이후 입력될 텍스트)에 문자/문단 서식을 적용합니다.

```python
# 문자 서식
app.set_charshape(
    bold=True,
    italic=True,
    underline_type=1,      # 1=single
    text_color="#FF0000",  # 빨강
    height=2400,           # 24pt
)

# 문단 서식
app.set_parashape(
    align_type=3,     # 1=left, 2=right, 3=center, 4=justify
    line_spacing=200, # 200%
)

# 현재 커서 위치의 값 읽기
cs = app.get_charshape()
print(cs.Bold, cs.Height, cs.TextColor)
```

::: {.callout-tip}
모든 속성은 `snake_case` 로 기입하거나 HWP 원본 `PascalCase` 둘 다 받습니다. `bold=True` 와 `Bold=1` 는 같은 뜻.
:::

## 6-2. 서식 자동 리셋 — `styled_text` · `charshape_scope` · `parashape_scope`

`set_charshape(bold=True)` 를 호출하면 이후 입력되는 모든 텍스트가 굵어집니다. 다음 입력에 영향을 안 주려면 `set_charshape(bold=False)` 로 되돌려야 합니다. 이런 on/off 쌍을 매번 작성하는 대신, hwpapi 는 **snapshot + restore** 패턴으로 서식을 자동 스코프화하는 3가지 헬퍼를 제공합니다.

### 동작 방식 — Snapshot + Restore

1. 진입 시 현재 `CharShape` / `ParaShape` 를 통째로 **snapshot** (`.clone()`)
2. 텍스트를 현재 상태로 삽입
3. 그 구간을 선택해서 요청한 서식만 적용
4. 커서를 끝으로 옮기고 **snapshot 으로 완전 복원**

이 방식은 `shade_color` (형광펜) 처럼 "기본값으로 되돌리기 어려운" 속성도 원래 값으로 정확히 복원합니다. 블록 이후 커서에는 어떤 서식 상태도 남지 않습니다.

### `styled_text(text, **fmt)` — 한 조각 텍스트

```python
app.insert_text("앞 — ")
app.styled_text("중요 부분만 굵고 빨강", bold=True, text_color="#FF0000")
app.insert_text(" — 뒤 (기본 서식)")

# 형광펜도 안전하게
app.insert_text("형광펜: ")
app.styled_text("노란 하이라이트", shade_color="#FFFF00")
app.insert_text(" — 이후는 음영 없음")
```

### `charshape_scope(**fmt)` — 여러 줄에 걸친 CharShape 블록

```python
with app.charshape_scope(bold=True, text_color="#0000FF"):
    app.insert_text("여러 insert 호출에 걸친\n")
    app.insert_text("파랑 굵은 글씨 블록\n")

app.insert_text("블록 종료 후 기본 서식 복원")
```

### `parashape_scope(**fmt)` — 문단 서식 블록

정렬, 줄간격, 들여쓰기 등 문단 레벨 서식을 블록 범위에만 적용:

```python
with app.parashape_scope(align_type=3, line_spacing=200):
    app.insert_text("가운데 정렬 + 200% 줄간격\n")
    app.insert_text("여러 문단도 OK\n")

app.insert_text("이후 문단은 원래 정렬/줄간격\n")
```

### 지원하는 서식

| 영역 | 속성 |
|:---|:---|
| **CharShape** (`styled_text`, `charshape_scope`) | `bold`, `italic`, `underline_type`, `strike_out_type`, `super_script`, `sub_script`, `text_color`, `shade_color` (형광펜), `height` (크기), `spacing_hangul` 등 |
| **ParaShape** (`parashape_scope`) | `align_type`, `line_spacing`, `indentation`, `left_margin`, `right_margin` 등 |

::: {.callout-tip}
**3가지 중 어느 것을 쓸까?**

- **단일 구(phrase) 하이라이트** → `styled_text("...", **fmt)`
- **여러 줄/여러 호출에 같은 문자 서식** → `with app.charshape_scope(**fmt):`
- **문단 정렬/줄간격 등 문단 속성** → `with app.parashape_scope(**fmt):`
:::

## 7. 텍스트 가져오기

현재 줄, 선택 영역, 전체 문서 등 범위를 지정해서 텍스트를 읽을 수 있습니다.

```python
from hwpapi import constants as const

# 기본: 현재 줄만
line = app.get_text()

# 전체 문서
full = app.get_text(
    spos=const.ScanStartPosition.Document,
    epos=const.ScanEndPosition.Document,
)

# 선택 영역
selected = app.get_selected_text()
```

## 종합 예제

지금까지 배운 내용을 조합한 5줄짜리 자동화 예:

```python
from hwpapi.core import App

app = App()
app.insert_text("자동화 문서\n\n이것은 hwpapi로 만든 문서입니다.\n")
app.move.top_of_file()
app.select_text()  # 첫 줄 선택
app.set_charshape(bold=True, height=2400)  # 제목 서식
app.save("test.hwp")
```

## 다음으로

- [찾기와 바꾸기](02_find_and_replace.html) — 검색과 치환의 다양한 옵션
- [기능 둘러보기](03_feature_tour.html) — 모든 기능을 카테고리별로
- 실전 사례 — [회의록](04_usecase_meeting_minutes.html), [일괄 편집](05_usecase_bulk_edit.html), [데이터→표](06_usecase_data_to_table.html), [보고서](07_usecase_report_generation.html)